# AE Pretraining Ablation Runner

Runs Stage 1 and optional Stage 1B experiments only. This notebook compares AE reconstruction quality before spending time on Stage 3/4/5 downstream runs.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.training.train_autoencoder import train_from_config, train_stage1b_from_config
from atlas_free_cnn.pipeline_outputs import create_full_pipeline_run_dir, write_table


In [ ]:
def split_dir_has_jsonl(path: Path) -> bool:
    return all((path / name).exists() for name in ["train.jsonl", "val.jsonl", "test.jsonl"])


HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
LOCAL_UNIFIED_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild"
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / "splits"
LOCAL_PACK_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn_rebuild"


def hf_download_first_available(filenames, local_dir: Path) -> Path:
    from huggingface_hub import hf_hub_download
    local_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    for filename in filenames:
        try:
            path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=filename,
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
            )
            return Path(path)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise FileNotFoundError("Could not download any candidate from HF:\n" + "\n".join(errors))


def ensure_hf_unified_splits() -> Path:
    print(f"Downloading atlas-free CNN split JSONLs from Hugging Face: {HF_DATASET_REPO}")
    LOCAL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "val", "test"]:
        downloaded = hf_download_first_available(
            [f"splits/{split}.jsonl", f"unified_jsonl_rebuild/splits/{split}.jsonl", f"{split}.jsonl"],
            LOCAL_UNIFIED_CACHE_DIR,
        )
        target = LOCAL_SPLIT_DIR / f"{split}.jsonl"
        if downloaded.resolve() != target.resolve():
            shutil.copy2(downloaded, target)
    for name in ["train_map_ids.json", "val_map_ids.json", "test_map_ids.json"]:
        try:
            downloaded = hf_download_first_available(
                [f"splits/{name}", f"unified_jsonl_rebuild/splits/{name}", name],
                LOCAL_UNIFIED_CACHE_DIR,
            )
            target = LOCAL_SPLIT_DIR / name
            if downloaded.resolve() != target.resolve():
                shutil.copy2(downloaded, target)
        except Exception as exc:
            print(f"Optional split sidecar not downloaded ({name}): {exc}")
    try:
        downloaded_volume = hf_download_first_available(
            ["atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn/atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn_rebuild/atlas_free_cnn_volumes.pt"],
            LOCAL_PACK_DIR,
        )
        target_volume = LOCAL_PACK_DIR / "atlas_free_cnn_volumes.pt"
        if downloaded_volume.resolve() != target_volume.resolve():
            try:
                if target_volume.exists() or target_volume.is_symlink():
                    target_volume.unlink()
                os.symlink(downloaded_volume, target_volume)
            except Exception:
                shutil.copy2(downloaded_volume, target_volume)
        print("Volume tensor available at:", target_volume)
    except Exception as exc:
        print("WARNING: split JSONLs downloaded, but volume tensor was not prepared:", exc)
        print("Training will fail unless tensor_path values inside JSONL resolve to an accessible tensor file.")
    return LOCAL_SPLIT_DIR


def discover_unified_split_dir() -> Path:
    override = os.environ.get("NEUROVLM_UNIFIED_SPLIT_DIR")
    candidates = []
    if override:
        candidates.append(Path(override))
    candidates.extend([
        REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "cache/unified_jsonl/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/unified_jsonl/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "data_ale_3dcnn/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_ale_3dcnn/unified_jsonl/splits",
    ])
    for candidate in candidates:
        if split_dir_has_jsonl(candidate):
            return candidate
    try:
        hf_split_dir = ensure_hf_unified_splits()
        if split_dir_has_jsonl(hf_split_dir):
            return hf_split_dir
    except Exception as exc:
        hf_error = exc
    else:
        hf_error = None
    checked = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find unified dataset split JSONL files locally, and Hugging Face fallback did not produce them. Expected train.jsonl, val.jsonl, and test.jsonl in one of:\n"
        f"{checked}\n\n"
        f"HF dataset repo tried: {HF_DATASET_REPO}\n"
        f"HF fallback error: {hf_error}\n\n"
        "If your splits are elsewhere, set os.environ['NEUROVLM_UNIFIED_SPLIT_DIR'] to that splits directory before running this cell."
    )

UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Train JSONL:", TRAIN_JSONL)

# quick: short smoke/iteration run; full: real AE ablation run.
AE_ABLATION_MODE = "quick"  # quick | full
SELECTED_AE_VARIANTS = "all"  # "all" or list like ["mixed_baseline_raw_mse", "mixed_balanced_raw_mse"]
AE_CHECKPOINT_SELECTION = "best_top5_dice"
RUN_STAGE1B = False  # set True after a mixed checkpoint looks worth fine-tuning
RESUME_COMPLETED = True

AE_EPOCHS = 2 if AE_ABLATION_MODE == "quick" else 300
STAGE1B_EPOCHS = 2 if AE_ABLATION_MODE == "quick" else 100
MAX_TRAIN_BATCHES = 20 if AE_ABLATION_MODE == "quick" else None
MAX_VAL_BATCHES = 5 if AE_ABLATION_MODE == "quick" else None
TRAIN_METRIC_BATCHES = 2 if AE_ABLATION_MODE == "quick" else 8
VAL_METRIC_BATCHES = 2 if AE_ABLATION_MODE == "quick" else 16
AE_BATCH_CANDIDATES = [384, 256, 192, 128, 96, 64, 48, 32, 16]
AE_PREFLIGHT_RESERVE_GB = 18.0 if AE_ABLATION_MODE == "quick" else 12.0
FINAL_EVAL = AE_ABLATION_MODE == "full"
SAVE_PLOTS = AE_ABLATION_MODE == "full"

ABLATION_OUTPUT_DIR = DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation" if "DRIVE_ROOT" in globals() else Path("runs_atlas_free_cnn_ae_ablation")
paths = create_full_pipeline_run_dir(ABLATION_OUTPUT_DIR, prefix="ae_ablation")
RUN_DIR = Path(paths["run_dir"])
print("Run directory:", RUN_DIR)
print({
    "AE_ABLATION_MODE": AE_ABLATION_MODE,
    "AE_EPOCHS": AE_EPOCHS,
    "MAX_TRAIN_BATCHES": MAX_TRAIN_BATCHES,
    "MAX_VAL_BATCHES": MAX_VAL_BATCHES,
    "AE_BATCH_CANDIDATES": AE_BATCH_CANDIDATES,
    "AE_PREFLIGHT_RESERVE_GB": AE_PREFLIGHT_RESERVE_GB,
    "FINAL_EVAL": FINAL_EVAL,
    "SAVE_PLOTS": SAVE_PLOTS,
    "RUN_STAGE1B": RUN_STAGE1B,
})


In [ ]:
recipes = [
    {
        "ae_variant": "mixed_baseline_raw_mse",
        "ae_training_recipe": "baseline_raw_mse",
        "source_sampling": "natural",
        "loss": {"type": "raw_mse", "lambda_foreground": 0.0, "lambda_topk": 0.0, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_val_loss",
    },
    {
        "ae_variant": "mixed_balanced_raw_mse",
        "ae_training_recipe": "mixed_balanced_raw_mse",
        "source_sampling": "balanced",
        "loss": {"type": "raw_mse", "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
    {
        "ae_variant": "mixed_balanced_hybrid_loss",
        "ae_training_recipe": "mixed_balanced_hybrid_loss",
        "source_sampling": "balanced",
        "loss": {"type": "hybrid_recon", "lambda_foreground": 0.10, "lambda_topk": 0.05, "topk_percent": 5, "prediction_activation": "none"},
        "checkpoint_selection_metric": "best_top5_dice",
    },
]

In [ ]:
import gc
import torch

results = []
selected = {r["ae_variant"]: r for r in recipes}
if SELECTED_AE_VARIANTS == "all":
    run_recipes = recipes
else:
    run_recipes = [selected[name] for name in SELECTED_AE_VARIANTS]


def clear_cuda_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def is_cuda_oom(exc: BaseException) -> bool:
    text = str(exc).lower()
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "cuda out of memory" in text or "outofmemoryerror" in text


def train_with_batch_fallback(base_cfg, candidates):
    candidates = [int(v) for v in candidates]
    last_exc = None
    for max_batch in candidates:
        clear_cuda_memory()
        cfg = dict(base_cfg)
        cfg["batch_candidates"] = [v for v in candidates if v <= max_batch]
        cfg["max_batch_size"] = max_batch
        cfg["batch_size"] = min(int(cfg.get("batch_size", 64)), max_batch)
        cfg["preflight_vram_reserve_gb"] = AE_PREFLIGHT_RESERVE_GB
        print(f"Trying max AE batch size {max_batch}; candidates={cfg['batch_candidates']}")
        try:
            return train_from_config(cfg)
        except Exception as exc:
            if not is_cuda_oom(exc):
                raise
            last_exc = exc
            print(f"OOM at max_batch_size={max_batch}. Falling back to next smaller candidate.")
            clear_cuda_memory()
    raise RuntimeError(f"All AE batch candidates failed: {candidates}") from last_exc

for recipe in run_recipes:
    out_dir = RUN_DIR / "01_stage1_ae_pretraining" / recipe["ae_variant"]
    best_checkpoint = out_dir / "checkpoints" / "best_cnn_autoencoder.pt"
    if RESUME_COMPLETED and best_checkpoint.exists():
        print(f"Skipping completed AE variant {recipe['ae_variant']}: {best_checkpoint}")
        results.append({"ae_variant": recipe["ae_variant"], "checkpoint_dir": str(out_dir / "checkpoints"), "best_checkpoint": str(best_checkpoint)})
        continue

    print(f"Running AE variant: {recipe['ae_variant']} ({AE_EPOCHS} epochs, mode={AE_ABLATION_MODE})")
    base_cfg = {
        "train_jsonl": TRAIN_JSONL,
        "val_jsonl": VAL_JSONL,
        "test_jsonl": TEST_JSONL,
        "output_dir": str(out_dir),
        "checkpoint_dir": str(out_dir / "checkpoints"),
        "data_mode": "mixed",
        "target_shape": [36, 45, 38],
        "model": {"latent_dim": 384, "base_channels": 64, "num_blocks": 4, "dropout": 0.1, "norm": "group", "pooling": "max"},
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "amp": True,
        "gradient_clipping": 1.0,
        "batch_size": 64,
        "epochs": AE_EPOCHS,
        "max_train_batches": MAX_TRAIN_BATCHES,
        "max_val_batches": MAX_VAL_BATCHES,
        "train_metric_batches": TRAIN_METRIC_BATCHES,
        "val_metric_batches": VAL_METRIC_BATCHES,
        "final_eval": FINAL_EVAL,
        "save_plots": SAVE_PLOTS,
        "progress": True,
        **recipe,
    }
    result = train_with_batch_fallback(base_cfg, AE_BATCH_CANDIDATES)
    results.append({"ae_variant": recipe["ae_variant"], "checkpoint_dir": result["checkpoint_dir"], "best_checkpoint": result["best_checkpoint"]})
    clear_cuda_memory()
results


In [ ]:
if RUN_STAGE1B and results:
    seed_checkpoint = results[-1]["best_checkpoint"]
    print("Stage 1B seed checkpoint:", seed_checkpoint)
    for mode, domain in [("mixed_pretrain_to_pubmed", "pubmed"), ("mixed_pretrain_to_statmaps", "statmaps")]:
        out_dir = RUN_DIR / "02_stage1b_ae_finetuning" / domain
        best_checkpoint = out_dir / "checkpoints" / "best_cnn_autoencoder.pt"
        if RESUME_COMPLETED and best_checkpoint.exists():
            print(f"Skipping completed Stage 1B {domain}: {best_checkpoint}")
            results.append({"ae_variant": "mixed_to_pubmed" if domain == "pubmed" else "mixed_to_statmaps", "checkpoint_dir": str(out_dir / "checkpoints"), "best_checkpoint": str(best_checkpoint)})
            continue
        cfg = {
            "stage1b_mode": mode,
            "mixed_pretrain_checkpoint": seed_checkpoint,
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "output_dir": str(out_dir),
            "checkpoint_dir": str(out_dir / "checkpoints"),
            "ae_variant": "mixed_to_pubmed" if domain == "pubmed" else "mixed_to_statmaps",
            "lr": 1e-4,
            "freeze_mode": "none",
            "epochs": STAGE1B_EPOCHS,
            "max_train_batches": MAX_TRAIN_BATCHES,
            "max_val_batches": MAX_VAL_BATCHES,
            "train_metric_batches": TRAIN_METRIC_BATCHES,
            "val_metric_batches": VAL_METRIC_BATCHES,
            "final_eval": FINAL_EVAL,
            "save_plots": SAVE_PLOTS,
            "loss": {"type": "raw_mse", "prediction_activation": "none"},
        }
        ft_result = train_stage1b_from_config(cfg)
        results.append({"ae_variant": cfg["ae_variant"], "checkpoint_dir": ft_result["checkpoint_dir"], "best_checkpoint": ft_result["best_checkpoint"]})
elif RUN_STAGE1B:
    print("Stage 1B requested, but no Stage 1 results are available.")
else:
    print("Stage 1B not requested.")


In [ ]:
summary_rows = []
for row in results:
    metrics_path = Path(row["checkpoint_dir"]).parent / "metrics" / "reconstruction_summary_by_source.csv"
    metrics = {}
    if metrics_path.exists():
        with metrics_path.open(newline="") as f:
            for m in csv.DictReader(f):
                if m.get("split") in {"val", "test"} and m.get("source_detail") == "ALL_DETAILS" and m.get("source") in {"pubmed", "neurovault", "nilearn"}:
                    src = m["source"]
                    metrics[f"{src}_spatial_corr"] = m.get("spatial_corr", "")
                    metrics[f"{src}_top5_dice"] = m.get("top5_dice", "")
    values = []
    for key, value in metrics.items():
        try:
            values.append(float(value))
        except Exception:
            pass
    ranking_score = sum(values) / len(values) if values else ""
    summary_rows.append({**row, **metrics, "ranking_score": ranking_score, "metrics_path": str(metrics_path), "recommendation_note": "Prioritize spatial_corr/top5_dice by source; do not rank by MSE alone."})
summary_rows = sorted(summary_rows, key=lambda r: float(r["ranking_score"]) if r["ranking_score"] != "" else -999, reverse=True)
write_table(RUN_DIR / "07_final_comparison" / "best_checkpoints_to_inspect.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "final_summary_table.csv", summary_rows)
write_table(RUN_DIR / "07_final_comparison" / "ae_ablation_leaderboard.csv", summary_rows)
print("AE ablation summary written to", RUN_DIR / "07_final_comparison")
summary_rows
